In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

df = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "regime_model_dataset.csv",
    parse_dates=["Date"],
)

print(df.shape)
print(df["Date"].min(), df["Date"].max())

(1171, 72)
2016-01-31 00:00:00 2025-10-31 00:00:00


In [2]:
baseline_features = [
    "mom_1m",
    "mom_3m",
    "mom_6m",
    "mom_12m",
    "vol_3m",
    "vol_6m",
    "vol_12m",
    "drawdown_12m",

    "spy_mom_1m",
    "spy_mom_3m",
    "spy_mom_6m",
    "spy_mom_12m",
    "spy_vol_3m",
    "spy_vol_6m",
    "spy_vol_12m",
    "spy_drawdown_12m",

    "DGS10",
    "DGS2",
    "DFF",
    "VIXCLS",
    "yield_spread_10y_2y",
    "DGS10_change_1m",
    "DGS2_change_1m",
    "DFF_change_1m",
    "VIXCLS_change_1m",
    "yield_spread_10y_2y_change_1m",
]

In [3]:
advanced_market_features = [
    "beta_6m",
    "beta_12m",
    "corr_spy_6m",
    "corr_spy_12m",
    "mom_3m_rank_pct",
    "mom_6m_rank_pct",
    "mom_12m_rank_pct",
    "vol_3m_rank_pct",
    "vol_6m_rank_pct",
    "vol_12m_rank_pct",
    "sector_return_dispersion",
]

In [4]:
new_macro_features = [
    "BAA10Y",
    "BAA10Y_change_1m",
    "DCOILWTICO",
    "DCOILWTICO_change_1m",
    "CPI_YOY_lag1",
    "CPI_YOY_change_1m_lag1",
    "UNRATE_lag1",
    "UNRATE_change_1m_lag1",
]

In [5]:
regime_features = [
    "high_vix_regime",
    "inverted_yield_curve",
    "rising_rate_regime",
    "credit_stress_regime",
    "rising_inflation_regime",
    "rising_unemployment_regime",
    "momentum_x_high_vix",
    "beta_x_high_vix",
    "momentum_x_inverted_curve",
    "momentum_x_credit_stress",
]

In [6]:
regime_aware_features = (
    baseline_features
    + advanced_market_features
    + new_macro_features
    + regime_features
)

In [7]:
train = df[
    df["Date"] <= "2022-12-31"
].copy()

test = df[
    df["Date"] >= "2023-01-01"
].copy()

print("Train:", train.shape)
print("Test:", test.shape)

print()
print(
    "Test range:",
    test["Date"].min(),
    "to",
    test["Date"].max(),
)

Train: (831, 72)
Test: (340, 72)

Test range: 2023-01-31 00:00:00 to 2025-10-31 00:00:00


In [8]:
X_train = train[
    regime_aware_features + ["Ticker"]
]

y_train = train["next_excess_return"]

X_test = test[
    regime_aware_features + ["Ticker"]
]


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            regime_aware_features,
        ),
        (
            "ticker",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            ["Ticker"],
        ),
    ]
)

regime_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0)),
])

regime_model.fit(
    X_train,
    y_train,
)

test["regime_prediction"] = (
    regime_model.predict(X_test)
)

In [9]:
results = []

for date, month in test.groupby("Date"):

    # -------------------------
    # 12-month momentum
    # -------------------------

    momentum_top3 = (
        month
        .sort_values(
            "mom_12m",
            ascending=False,
        )
        .head(3)
    )

    results.append({
        "Date": date,
        "strategy": "12m_momentum",
        "spearman": (
            month["mom_12m"]
            .corr(
                month["next_excess_return"],
                method="spearman",
            )
        ),
        "precision_at_3":
            momentum_top3["target"].mean(),
        "top3_return":
            momentum_top3["next_etf_return"].mean(),
        "spy_return":
            momentum_top3["next_spy_return"].iloc[0],
        "top3_excess_return":
            momentum_top3[
                "next_excess_return"
            ].mean(),
    })

    # -------------------------
    # Regime-aware ML
    # -------------------------

    ml_top3 = (
        month
        .sort_values(
            "regime_prediction",
            ascending=False,
        )
        .head(3)
    )

    results.append({
        "Date": date,
        "strategy": "regime_aware_ml",
        "spearman": (
            month["regime_prediction"]
            .corr(
                month["next_excess_return"],
                method="spearman",
            )
        ),
        "precision_at_3":
            ml_top3["target"].mean(),
        "top3_return":
            ml_top3["next_etf_return"].mean(),
        "spy_return":
            ml_top3["next_spy_return"].iloc[0],
        "top3_excess_return":
            ml_top3[
                "next_excess_return"
            ].mean(),
    })


test_results = pd.DataFrame(results)

In [10]:
test_summary = (
    test_results
    .groupby("strategy")
    .agg(
        mean_spearman=(
            "spearman",
            "mean",
        ),
        median_spearman=(
            "spearman",
            "median",
        ),
        precision_at_3=(
            "precision_at_3",
            "mean",
        ),
        mean_monthly_return=(
            "top3_return",
            "mean",
        ),
        mean_monthly_excess_return=(
            "top3_excess_return",
            "mean",
        ),
        positive_excess_months=(
            "top3_excess_return",
            lambda x: (x > 0).mean(),
        ),
    )
)

test_summary

,mean_spearman,median_spearman,precision_at_3,mean_monthly_return,mean_monthly_excess_return,positive_excess_months
strategy,,,,,,
12m_momentum,0.027094,0.018182,0.431373,0.012252,-0.004810,0.500000
regime_aware_ml,0.094831,0.151515,0.450980,0.014131,-0.002931,0.441176


In [11]:
performance_results = []

for strategy, strategy_df in test_results.groupby(
    "strategy"
):

    strategy_df = (
        strategy_df
        .sort_values("Date")
        .copy()
    )

    strategy_growth = (
        1 + strategy_df["top3_return"]
    ).cumprod()

    spy_growth = (
        1 + strategy_df["spy_return"]
    ).cumprod()

    cumulative_return = (
        strategy_growth.iloc[-1] - 1
    )

    spy_cumulative_return = (
        spy_growth.iloc[-1] - 1
    )

    monthly_returns = (
        strategy_df["top3_return"]
    )

    annualized_return = (
        strategy_growth.iloc[-1]
        ** (12 / len(strategy_df))
        - 1
    )

    annualized_volatility = (
        monthly_returns.std()
        * np.sqrt(12)
    )

    sharpe = (
        annualized_return
        / annualized_volatility
        if annualized_volatility != 0
        else np.nan
    )

    running_peak = (
        strategy_growth.cummax()
    )

    drawdown = (
        strategy_growth / running_peak - 1
    )

    max_drawdown = drawdown.min()

    performance_results.append({
        "strategy": strategy,
        "cumulative_return":
            cumulative_return,
        "annualized_return":
            annualized_return,
        "annualized_volatility":
            annualized_volatility,
        "sharpe":
            sharpe,
        "max_drawdown":
            max_drawdown,
        "spy_cumulative_return":
            spy_cumulative_return,
    })


performance_summary = pd.DataFrame(
    performance_results
)

performance_summary

,strategy,cumulative_return,annualized_return,annualized_volatility,sharpe,max_drawdown,spy_cumulative_return
0,12m_momentum,0.477298,0.147657,0.133360,1.107205,-0.088984,0.743832
1,regime_aware_ml,0.566843,0.171744,0.144931,1.184999,-0.114039,0.743832
